# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a walkthrough for loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all RecordSets and their field @id's

record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all main record sets

# Get the @id of each RecordSet
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns (fields as @id) of the first record set
if record_set_ids:
    print(f"Columns in first RecordSet ({record_set_ids[0]}):")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. For this example, we'll select a numeric field if available and perform basic processing.

In [ ]:
# Select a record set and a numeric field for EDA

import numpy as np

# For demonstration, automatically select the first record set with a numeric field
selected_record_set_id = None
numeric_field_id = None

for rs in dataset.record_sets:
    df = dataframes.get(rs.id)
    if df is not None and not df.empty:
        # Find the first field with an integer or float-like dtype
        for field in rs.fields:
            if field.data_type in ('http://schema.org/Integer', 'http://schema.org/Float', 'Float', 'Integer', 'Number', 'schema:Integer', 'schema:Float', 'schema:Number'):
                if field.id in df.columns:
                    # Try casting to numeric to confirm
                    try:
                        df[field.id] = pd.to_numeric(df[field.id], errors='coerce')
                        if df[field.id].notna().sum() > 0:
                            selected_record_set_id = rs.id
                            numeric_field_id = field.id
                            break
                    except Exception:
                        continue
        if selected_record_set_id:
            break

if selected_record_set_id and numeric_field_id:
    print(f"Using record set {selected_record_set_id} and numeric field {numeric_field_id}")
    df = dataframes[selected_record_set_id]
    # Filter for numeric_field > threshold (use mean as threshold for small dataset)
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by a non-numeric field if available
    group_field_id = None
    for field in dataset.record_sets_by_id[selected_record_set_id].fields:
        if field.id != numeric_field_id and field.id in df.columns:
            # Assume fields with string values and less than, e.g., 10 unique values as candidates
            if df[field.id].dtype == object and df[field.id].nunique() < 10:
                group_field_id = field.id
                break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Could not locate a numeric field for EDA.")

## 5. Visualization

Visualize the distribution of a selected numeric field in the selected record set.

_If no numeric field was found above, this cell will not plot._

In [ ]:
import matplotlib.pyplot as plt

if selected_record_set_id and numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(7, 4))
    plt.hist(filtered_df[numeric_field_id].dropna(), bins=10, color="skyblue", edgecolor="k")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset: *Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution*.

- We reviewed the available record sets and their fields via their Croissant `@id`s.
- We loaded data into DataFrames, filtered and normalized a numeric field, and grouped by a categorical field (where available).
- We visualized the distribution of the numeric attribute.

This notebook provides a solid foundation for more advanced clinical outcome analysis, such as statistical hypothesis testing, model training, or detailed subgroup exploration using the rich, schema-driven Croissant structure.